# Notebook — Carga de conocimiento MedProduct en Qdrant 

Versión arreglada para incluir correctamente:

- `MEDPRODUCT_OBJECIONES_PHASE2_CANONICAL` como archivo vectorizable para Qdrant.
- `RETRIEVAL_OPTIMIZATION` como reglas FASE 5, sin indexarlas como chunks clínicos.

Flujo:

1. Configurar rutas y variables de entorno.
2. Preparar el archivo de objeciones como `*_VECTOR_INDEX.jsonl` si viene en `.json`.
3. Cargar reglas FASE 5 desde `OUTPUTS/RETRIEVAL_OPTIMIZATION`.
4. Cargar JSONL vectorizables.
5. Crear embeddings.
6. Conectar con Qdrant.
7. Crear/recrear colección.
8. Subir chunks con metadata.
9. Crear índices de payload.
10. Probar búsqueda optimizada.
11. Construir contexto para enviarlo a un LLM/avatar.


### 1. Configuramos rutas y parámetros

Modifica estas rutas si cambias la ubicación del proyecto o del archivo `.env`.


In [ ]:
import os
from pathlib import Path

# Carpeta raíz del proyecto
BASE_DIR = Path(os.getenv("BASE_DIR", Path.cwd()))

# Subcarpetas esperadas
OUTPUTS_DIR = BASE_DIR / "OUTPUTS"
VECTOR_INDEX_DIR = OUTPUTS_DIR / "VECTOR_INDEX"
PHASE5_DIR = OUTPUTS_DIR / "RETRIEVAL_OPTIMIZATION"

# Archivo .env con QDRANT_URL, QDRANT_API_KEY y otras claves si existen
ENV_PATH = Path(os.getenv("ENV_PATH", BASE_DIR / ".env"))

# Nombre de la colección en Qdrant
COLLECTION_NAME = "medproduct_medical_rag"

# Modelo local de embeddings. Dimensión esperada: 384
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Si True, borra la colección existente y la recrea desde cero.
# Para reconstruir toda la colección con los nuevos archivos, déjalo en True.
# Si quieres añadir sin borrar, ponlo en False, pero usa IDs estables para no sobrescribir.
RECREATE_COLLECTION = True

# Crear carpetas si no existen
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_INDEX_DIR.mkdir(parents=True, exist_ok=True)
PHASE5_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR existe:", BASE_DIR.exists(), "→", BASE_DIR)
print("OUTPUTS_DIR existe:", OUTPUTS_DIR.exists(), "→", OUTPUTS_DIR)
print("VECTOR_INDEX_DIR existe:", VECTOR_INDEX_DIR.exists(), "→", VECTOR_INDEX_DIR)
print("PHASE5_DIR existe:", PHASE5_DIR.exists(), "→", PHASE5_DIR)
print("ENV_PATH existe:", ENV_PATH.exists(), "→", ENV_PATH)


### 2. Instalamos dependencias

Ejecuta esta celda solo si el entorno no tiene todavía estas librerías instaladas.


In [ ]:
# Ejecutar solo si hace falta
# !pip install sentence-transformers qdrant-client python-dotenv


### 3. Importamos librerías y cargamos credenciales desde `.env`

No se deben escribir API keys directamente en el notebook. Las claves se leen desde el archivo `.env`.


In [ ]:
import os
import json
import uuid
import shutil
from dotenv import load_dotenv

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    Filter,
    FieldCondition,
    MatchText,
    PayloadSchemaType,
)
from sentence_transformers import SentenceTransformer

load_dotenv(ENV_PATH, override=True)

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("QDRANT_URL cargada:", QDRANT_URL is not None)
print("QDRANT_API_KEY cargada:", QDRANT_API_KEY is not None)
print("GOOGLE_API_KEY cargada:", GOOGLE_API_KEY is not None)
print("OPENAI_API_KEY cargada:", OPENAI_API_KEY is not None)

if not QDRANT_URL or not QDRANT_API_KEY:
    raise ValueError("Faltan QDRANT_URL o QDRANT_API_KEY en el archivo .env")


### 4. Preparamos objeciones y cargamos reglas FASE 5


In [ ]:
TARGET_OBJECIONES_JSONL = VECTOR_INDEX_DIR / "MEDPRODUCT_OBJECIONES_PHASE2_CANONICAL_VECTOR_INDEX.jsonl"


def _extract_records_from_json(data):
    """Extrae una lista de registros desde varios formatos JSON posibles."""
    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ["records", "chunks", "canonical_chunks", "vector_index", "items", "data"]:
            if key in data and isinstance(data[key], list):
                return data[key]

    raise ValueError(
        "No se ha encontrado una lista de registros en el JSON. "
        "Busqué: records, chunks, canonical_chunks, vector_index, items o data."
    )


def _record_text(record):
    """Obtiene el texto vectorizable de un registro."""
    text = record.get("text")

    if not text:
        content = record.get("content")
        if isinstance(content, dict):
            text = (
                content.get("normalized_claim")
                or content.get("exact_quote")
                or json.dumps(content, ensure_ascii=False)
            )
        elif isinstance(content, str):
            text = content

    if not text:
        text = record.get("normalized_claim") or record.get("exact_quote")

    if not text:
        raise ValueError(f"Registro sin texto vectorizable: {record.get('id') or record.get('chunk_id')}")

    return str(text).strip()


def _normalize_record_for_qdrant(record, idx):
    """Convierte un registro al formato esperado por el indexador: id, text, metadata."""
    record_id = record.get("id") or record.get("chunk_id") or f"MEDPRODUCT_OBJ_{idx:04d}"

    metadata = dict(record.get("metadata", {}))

    # Campos clínicos útiles que conviene conservar como payload.
    for key in [
        "chunk_id",
        "chunk_title",
        "chunk_type",
        "document_metadata",
        "classification_4_levels",
        "normalized_entities",
        "retrieval_metadata",
        "wiki_preparation",
    ]:
        if key in record:
            metadata[key] = record[key]

    # Campos derivados para facilitar filtros y trazabilidad.
    metadata.setdefault("source_package", "MEDPRODUCT_OBJECIONES_PHASE2_CANONICAL")
    metadata.setdefault("pipeline_phase", "PHASE_2_CANONICAL_JSON_GENERATION")

    if "document_metadata" in record and isinstance(record["document_metadata"], dict):
        dm = record["document_metadata"]
        metadata.setdefault("source_file", dm.get("source_file"))
        metadata.setdefault("page_start", dm.get("page_start"))
        metadata.setdefault("page_end", dm.get("page_end"))
        metadata.setdefault("section_title", dm.get("section_title"))

    if "content" in record and isinstance(record["content"], dict):
        content = record["content"]
        metadata.setdefault("claim_type", content.get("claim_type"))
        metadata.setdefault("evidence_level", content.get("evidence_level"))
        metadata.setdefault("safety_risk", content.get("safety_risk"))
        metadata.setdefault("confidence_score", content.get("confidence_score"))

    if "retrieval_metadata" in record and isinstance(record["retrieval_metadata"], dict):
        rm = record["retrieval_metadata"]
        metadata.setdefault("retrieval_priority", rm.get("retrieval_priority"))
        metadata.setdefault("reranking_policy", rm.get("reranking_policy"))

    return {
        "id": str(record_id),
        "text": _record_text(record),
        "metadata": metadata,
    }


# Buscar el JSON vectorizable original si existe.
source_candidates = sorted(BASE_DIR.rglob("MEDPRODUCT_OBJECIONES_VECTOR_INDEX.json"))

if TARGET_OBJECIONES_JSONL.exists():
    print("Ya existe el JSONL de objeciones:")
    print(TARGET_OBJECIONES_JSONL)

elif source_candidates:
    source_vector_file = source_candidates[0]
    print("Archivo vector de objeciones encontrado:")
    print(source_vector_file)

    with open(source_vector_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    records_raw = _extract_records_from_json(data)
    records_norm = [
        _normalize_record_for_qdrant(record, idx)
        for idx, record in enumerate(records_raw)
    ]

    with open(TARGET_OBJECIONES_JSONL, "w", encoding="utf-8") as f:
        for record in records_norm:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print("JSONL creado para Qdrant:")
    print(TARGET_OBJECIONES_JSONL)
    print("Registros de objeciones preparados:", len(records_norm))

else:
    print("No he encontrado MEDPRODUCT_OBJECIONES_VECTOR_INDEX.json.")
    print("Si ya tienes el .jsonl final dentro de OUTPUTS/VECTOR_INDEX, no pasa nada.")


# 4.2. Cargar reglas FASE 5 sin indexarlas en Qdrant

phase5_rules = {}
phase5_json_files = sorted(PHASE5_DIR.rglob("*.json"))

for file in phase5_json_files:
    with open(file, "r", encoding="utf-8") as f:
        # Clave legible con ruta relativa para evitar colisiones de nombres.
        key = str(file.relative_to(PHASE5_DIR)).replace("\\", "/")
        phase5_rules[key] = json.load(f)

print("\nReglas FASE 5 cargadas:", len(phase5_rules))
for key in phase5_rules:
    print("-", key)

print("\nNota: FASE 5 queda cargada como reglas del sistema, no como chunks vectorizados.")


### 4. Localizamos los archivos vectorizables

Para la base vectorial usamos los archivos `*_VECTOR_INDEX.jsonl`, no los Markdown de la wiki ni los ZIP de auditoría/retrieval.


In [ ]:
# Localizamos solo archivos .jsonl vectorizables.
# La FASE 5 no entra aquí porque no se indexa como contenido clínico.

vector_files = sorted(BASE_DIR.rglob("*VECTOR*.jsonl"))

print("Archivos VECTOR_INDEX encontrados:", len(vector_files))
for file in vector_files:
    print(file)

if not vector_files:
    raise FileNotFoundError(
        "No se han encontrado archivos *_VECTOR*.jsonl. "
        "Comprueba que el archivo está en OUTPUTS\\VECTOR_INDEX y termina en .jsonl."
    )


### Comprobación rápida de estructura esperada

Esta celda muestra si el archivo de objeciones está donde debe y recuerda que FASE 5 no se indexa como chunk.


In [ ]:
print("VECTOR_INDEX_DIR:", VECTOR_INDEX_DIR)
print("Archivos .jsonl dentro de VECTOR_INDEX:")
for file in sorted(VECTOR_INDEX_DIR.glob("*.jsonl")):
    print("-", file.name)

print("\nPHASE5_DIR:", PHASE5_DIR)
print("FASE 5 encontrada:", PHASE5_DIR.exists())
print("Reglas FASE 5 cargadas en memoria:", len(phase5_rules))


### 5. Revisamos la estructura de un chunk

Cada línea del JSONL debe tener normalmente:

- `id`: identificador del chunk.
- `text`: texto que se vectoriza.
- `metadata`: trazabilidad, tipo de fuente, autoridad, tipo de chunk, seguridad, etc.


In [ ]:
sample_file = vector_files[0]

with open(sample_file, "r", encoding="utf-8") as f:
    first_line = f.readline()
    sample = json.loads(first_line)

sample


### 6. Cargamos todos los chunks

Añadimos también dos campos auxiliares:

- `_source_file`: nombre del archivo JSONL de origen.
- `_source_folder`: carpeta/documento de origen.


In [ ]:
records = []

for file in vector_files:
    with open(file, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue

            item = json.loads(line)

            if "text" not in item or not str(item["text"]).strip():
                raise ValueError(f"Registro sin campo text en {file}, línea {line_number}")

            item["_source_file"] = file.name
            item["_source_folder"] = file.parent.name
            records.append(item)

print("Chunks cargados:", len(records))

if not records:
    raise ValueError("No se cargó ningún chunk.")

records[0]


### 7. Creamos embeddings

Usamos el campo `text` de cada chunk. El modelo seleccionado genera vectores de 384 dimensiones.


In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = [record["text"] for record in records]

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Shape de embeddings:", embeddings.shape)


### 8. Conectamos con Qdrant Cloud

La conexión usa `QDRANT_URL` y `QDRANT_API_KEY` cargadas desde `.env`.


In [ ]:
import requests

test_url = QDRANT_URL.rstrip("/") + "/collections"

response = requests.get(
    test_url,
    headers={"api-key": QDRANT_API_KEY},
    timeout=30,
)

print("URL probada:", test_url)
print("Status:", response.status_code)
print("Respuesta:", response.text[:500])

In [ ]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient

load_dotenv(override=True)

QDRANT_URL = os.getenv("QDRANT_URL", "").strip()
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "").strip()

print("QDRANT_URL:", repr(QDRANT_URL))
print("API KEY cargada:", bool(QDRANT_API_KEY))
print("API KEY longitud:", len(QDRANT_API_KEY))
print("API KEY final:", QDRANT_API_KEY[-6:])

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    check_compatibility=False,
    timeout=30,
)

collections = client.get_collections()
print("Conexión correcta.")
print(collections)

### 9. Creamos o recreamos la colección

La dimensión de la colección debe coincidir con la dimensión real de los embeddings. En este caso, normalmente será `384`.


**Nota:** si `RECREATE_COLLECTION = True`, la colección existente se borra y se vuelve a crear desde cero. Para no borrar datos ya cargados, ponlo en `False`.


In [ ]:
vector_size = embeddings.shape[1]

if RECREATE_COLLECTION and client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
    print("Colección anterior eliminada:", COLLECTION_NAME)

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=vector_size,
            distance=Distance.COSINE,
        ),
    )
    print("Colección creada:", COLLECTION_NAME)
else:
    print("La colección ya existe:", COLLECTION_NAME)

print("Dimensión vectorial:", vector_size)


### 10. Subimos los chunks a Qdrant

Cada punto incluye:

- vector del embedding;
- texto del chunk;
- metadata clínica/regulatoria;
- archivo y carpeta de origen.


In [ ]:
points = []

for i, record in enumerate(records):
    payload = {
        "text": record["text"],
        **record.get("metadata", {}),
        "_source_file": record.get("_source_file"),
        "_source_folder": record.get("_source_folder"),
        "_record_id": record.get("id"),
    }

    source_file = str(payload.get("_source_file") or "").lower()
    source_folder = str(payload.get("_source_folder") or "").lower()
    text = str(payload.get("text") or "")

    # ============================================================
    # Metadata por defecto si no venía en el .jsonl
    # ============================================================

    if "secure" in source_file or "secure" in source_folder:
        payload.setdefault("source_type", "clinical_trial_publication")
        payload.setdefault("authority_level", "LEVEL_3_PIVOTAL_CLINICAL_TRIAL")

    elif "prospecto" in source_file or "prospecto" in source_folder:
        payload.setdefault("source_type", "prospectus")
        payload.setdefault("authority_level", "LEVEL_1_REGULATORY_SOURCE")

    elif (
        "cima" in source_file
        or "aemps" in source_file
        or "cima" in source_folder
        or "aemps" in source_folder
    ):
        payload.setdefault("source_type", "regulatory_drug_database")
        payload.setdefault("authority_level", "LEVEL_1_REGULATORY_SOURCE")

    elif "pandora" in source_file or "pandora" in source_folder:
        payload.setdefault("source_type", "real_world_evidence")
        payload.setdefault("authority_level", "LEVEL_4_RWE_PRIMARY_SOURCE")

    elif "neptuno" in source_file or "neptuno" in source_folder:
        payload.setdefault("source_type", "real_world_evidence")
        payload.setdefault("authority_level", "LEVEL_4_RWE_PRIMARY_SOURCE")

    elif "objeciones" in source_file or "objeciones" in source_folder:
        payload.setdefault("source_type", "commercial_objection_handling")
        payload.setdefault("authority_level", "LEVEL_7_COMMERCIAL_POSITIONING_SUPPORT")

    elif (
        "gaziano" in source_file
        or "gaziano" in source_folder
        or "cost" in source_file
        or "cost" in source_folder
        or "pharmaco" in source_file
        or "pharmaco" in source_folder
        or "farmaco" in source_file
        or "farmaco" in source_folder
    ):
        payload.setdefault("source_type", "pharmacoeconomic_source")
        payload.setdefault("authority_level", "LEVEL_6_PHARMACOECONOMIC_SOURCE")

    else:
        payload.setdefault("source_type", "unknown")
        payload.setdefault("authority_level", "unknown")

    # ============================================================
    # Inferir chunk_type desde el texto si no venía separado
    # ============================================================

    if "CLINICAL_EVIDENCE_CHUNK" in text:
        payload.setdefault("chunk_type", "CLINICAL_EVIDENCE_CHUNK")

    elif "SAFETY_WARNING_CHUNK" in text:
        payload.setdefault("chunk_type", "SAFETY_WARNING_CHUNK")

    elif "CONTRAINDICATION" in text or "CONTRAINDICATIONS" in text:
        payload.setdefault("chunk_type", "CONTRAINDICATION_CHUNK")

    elif "DOSING" in text or "POSOLOGY" in text:
        payload.setdefault("chunk_type", "DOSING_CHUNK")

    elif "INTERACTION" in text or "INTERACTIONS" in text:
        payload.setdefault("chunk_type", "INTERACTION_CHUNK")

    elif "RWE" in text or "REAL_WORLD" in text:
        payload.setdefault("chunk_type", "RWE_CHUNK")

    elif "PHARMACOECONOMIC" in text or "QALY" in text:
        payload.setdefault("chunk_type", "PHARMACOECONOMIC_CHUNK")

    elif "POSITIONING" in text or "OBJECTION" in text:
        payload.setdefault("chunk_type", "POSITIONING_ARGUMENT_CHUNK")

    else:
        payload.setdefault("chunk_type", "general_medical_chunk")

    # ID estable para evitar sobrescrituras accidentales si se añade contenido sin recrear colección.
    stable_id_source = f'{payload.get("_source_file")}|{record.get("id", i)}|{i}'
    point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, stable_id_source))

    points.append(
        PointStruct(
            id=point_id,
            vector=embeddings[i].tolist(),
            payload=payload,
        )
    )

client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)

print("Chunks subidos a Qdrant:", len(points))

### 11. Creamos índices de payload para filtrar por metadata

Esto permite aplicar filtros sobre campos como `_source_file`, `authority_level`, `source_type` o `chunk_type`.


In [ ]:
def create_payload_index_if_needed(field_name, schema=PayloadSchemaType.TEXT):
    try:
        client.create_payload_index(
            collection_name=COLLECTION_NAME,
            field_name=field_name,
            field_schema=schema,
        )
        print("Índice creado:", field_name)
    except Exception as exc:
        # Si ya existe, Qdrant puede lanzar error. Lo dejamos registrado sin detener el notebook.
        print("Índice no creado o ya existente:", field_name, "→", str(exc)[:160])

for field in [
    "_source_file",
    "_source_folder",
    "_record_id",
    "authority_level",
    "source_type",
    "chunk_type",
    "source_package",
    "pipeline_phase",
    "retrieval_priority",
    "safety_risk",
    "evidence_level",
]:
    create_payload_index_if_needed(field)


### 12. Definimos la función de búsqueda optimizada

Esta función aplica la lógica práctica de FASE 5:

- búsqueda semántica por embeddings;
- filtros por documento cuando la pregunta menciona SECURE, NEPTUNO, PANDORA o Gaziano;
- filtros por autoridad regulatoria para seguridad, contraindicaciones, interacciones y poblaciones especiales;
- reranking manual para priorizar los chunks clínicamente más útiles.


In [ ]:
def search_medproduct(query, limit=5):
    query_vector = embedding_model.encode(
        query,
        normalize_embeddings=True,
    ).tolist()

    q = query.upper()
    query_filter = None

    # 1. Filtro por estudios concretos
    if "SECURE" in q:
        query_filter = Filter(
            must=[FieldCondition(key="_source_file", match=MatchText(text="SECURE"))]
        )

    elif "NEPTUNO" in q:
        query_filter = Filter(
            must=[FieldCondition(key="_source_file", match=MatchText(text="NEPTUNO"))]
        )

    elif "PANDORA" in q:
        query_filter = Filter(
            must=[FieldCondition(key="_source_file", match=MatchText(text="PANDORA"))]
        )

    elif "GAZIANO" in q or "COSTE" in q or "COSTE-EFECTIVIDAD" in q or "COSTE EFECTIVIDAD" in q:
        query_filter = Filter(
            must=[FieldCondition(key="_source_file", match=MatchText(text="GAZIANO"))]
        )

    # 2. Seguridad regulatoria: priorizar ficha técnica / prospecto / CIMA
    elif any(term in q for term in [
        "CONTRAINDIC", "INTERAC", "METOTREXATO", "IBUPROFENO",
        "RENAL", "HEMODIÁLISIS", "HEMODIALISIS", "EMBARAZO",
        "HEPÁTICA", "HEPATICA", "POMELO", "ALERGIA",
        "SACUBITRILO", "VALSARTÁN", "VALSARTAN",
        "HEMORRAGIA", "SANGRADO", "ANGIOEDEMA",
        "HIPERPOTASEMIA", "POTASIO", "TOS", "MIOPATÍA",
        "MIOPATIA", "RABDOMIOLISIS", "LACTOSA", "SOJA",
        "CACAHUETE"
    ]):
        query_filter = Filter(
            must=[FieldCondition(key="authority_level", match=MatchText(text="LEVEL_1"))]
        )

    # 3. Búsqueda en Qdrant
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=query_filter,
        limit=12,
        with_payload=True,
    )

    points = results.points

    # 4. Reranking específico para SECURE
    if "SECURE" in q:
        def secure_rank(point):
            text = point.payload.get("text", "").upper()
            score = point.score

            if "SECURE_CHUNK_005_PRIMARY_ENDPOINT_RESULT" in text:
                score += 0.60
            if "SECURE_CHUNK_006_KEY_SECONDARY_ENDPOINT_RESULT" in text:
                score += 0.55
            if "SECURE_CHUNK_007_ADHERENCE_RESULT" in text:
                score += 0.40
            if "SECURE_CHUNK_009_SAFETY_RESULT" in text:
                score += 0.30

            if "SECURE_CHUNK_004_ENDPOINT_DEFINITIONS" in text:
                score += 0.25
            if "SECURE_CHUNK_002_TRIAL_DESIGN_POPULATION" in text:
                score += 0.15

            if "SECURE_CHUNK_010_STATISTICAL_ANALYSIS" in text:
                score -= 0.15
            if "SECURE_CHUNK_011_LIMITATIONS" in text:
                score -= 0.20

            return score

        points = sorted(points, key=secure_rank, reverse=True)

    # 5. Reranking específico para NEPTUNO
    elif "NEPTUNO" in q:
        def neptuno_rank(point):
            text = point.payload.get("text", "").upper()
            score = point.score

            if "MACE" in text or "RECURRENT" in text or "EVENTOS" in text:
                score += 0.35
            if "PERSISTENCE" in text or "PERSISTENCIA" in text:
                score += 0.25
            if "LDL" in text or "BLOOD PRESSURE" in text or "PRESIÓN" in text or "PRESION" in text:
                score += 0.20
            if "LIMITATION" in text or "LIMITACIONES" in text:
                score -= 0.15

            return score

        points = sorted(points, key=neptuno_rank, reverse=True)

    # 6. Reranking específico para PANDORA
    elif "PANDORA" in q:
        def pandora_rank(point):
            text = point.payload.get("text", "").upper()
            score = point.score

            if "DISCHARGE" in text or "ALTA" in text:
                score += 0.30
            if "PERSISTENCE" in text or "PERSISTENCIA" in text or "76,5" in text:
                score += 0.30
            if "100MG/40MG/2,5MG" in text or "100/40/2,5" in text:
                score += 0.20
            if "ADJUVANT" in text or "COADYUVANTE" in text:
                score += 0.20
            if "LIMITATION" in text or "LIMITACIONES" in text:
                score -= 0.15
            if "RETRIEVAL_GOVERNANCE" in text:
                score -= 0.30

            return score

        points = sorted(points, key=pandora_rank, reverse=True)

    # 7. Reranking para coste-efectividad
    elif "GAZIANO" in q or "COSTE" in q or "COSTE-EFECTIVIDAD" in q or "COSTE EFECTIVIDAD" in q:
        def cost_rank(point):
            text = point.payload.get("text", "").upper()
            score = point.score

            if "DOMINANT" in text or "DOMINANTE" in text:
                score += 0.35
            if "QALY" in text or "QALYS" in text:
                score += 0.25
            if "SPANISH HEALTHCARE" in text or "ESPAÑA" in text or "SPAIN" in text:
                score += 0.20
            if "PROBABILISTIC" in text or "SENSITIVITY" in text or "SENSIBILIDAD" in text:
                score += 0.15

            return score

        points = sorted(points, key=cost_rank, reverse=True)

    # 8. Reranking para seguridad / regulación
    elif any(term in q for term in [
        "CONTRAINDIC", "INTERAC", "METOTREXATO", "IBUPROFENO",
        "RENAL", "HEMODIÁLISIS", "HEMODIALISIS", "EMBARAZO",
        "HEPÁTICA", "HEPATICA", "POMELO", "ALERGIA",
        "SACUBITRILO", "VALSARTÁN", "VALSARTAN",
        "HEMORRAGIA", "SANGRADO", "ANGIOEDEMA",
        "HIPERPOTASEMIA", "POTASIO", "TOS", "MIOPATÍA",
        "MIOPATIA", "RABDOMIOLISIS", "LACTOSA", "SOJA",
        "CACAHUETE"
    ]):
        def safety_rank(point):
            authority = str(point.payload.get("authority_level", "")).upper()
            chunk_type = str(point.payload.get("chunk_type", "")).upper()
            score = point.score

            if "LEVEL_1" in authority:
                score += 0.30
            if "CONTRAINDICATION" in chunk_type:
                score += 0.25
            if "SAFETY_WARNING" in chunk_type:
                score += 0.20
            if "DOSING" in chunk_type and ("RENAL" in q or "HEPÁTICA" in q or "HEPATICA" in q):
                score += 0.20

            return score

        points = sorted(points, key=safety_rank, reverse=True)

    return points[:limit]


### 13. Probamos el retrieval con preguntas de demo

Estas preguntas permiten validar que la base vectorial recupera las fuentes correctas para la demo interna.


In [ ]:
test_queries = [
    "¿Qué es MedProduct?",
    "¿Qué evidencia tiene SECURE?",
    "¿Qué aporta NEPTUNO?",
    "¿Qué dice PANDORA?",
    "¿Es coste-efectiva MedProduct?",
    "¿Puede usarse MedProduct en insuficiencia renal?",
    "¿Está contraindicado con metotrexato?",
    "¿Puede tomarse con ibuprofeno?",
    "¿Cómo se toma MedProduct?",
]

for query in test_queries:
    print("PREGUNTA:", query)

    results = search_medproduct(query, limit=3)

    for r in results:
        print("SCORE ORIGINAL:", r.score)
        print("SOURCE_FILE:", r.payload.get("_source_file"))
        print("SOURCE_FOLDER:", r.payload.get("_source_folder"))
        print("SOURCE_TYPE:", r.payload.get("source_type"))
        print("AUTHORITY:", r.payload.get("authority_level"))
        print("TIPO:", r.payload.get("chunk_type"))
        print("TEXTO:", r.payload.get("text")[:500])
        print("-" * 80)


### 14. Construimos contexto para enviarlo a un LLM

Esta función convierte los chunks recuperados en un bloque de contexto listo para enviarlo a Gemini, OpenAI u otro LLM.


In [ ]:
def build_context(query, limit=3):
    results = search_medproduct(query, limit=limit)
    context_blocks = []

    for i, r in enumerate(results, start=1):
        payload = r.payload

        block = f"""
[CHUNK {i}]
SOURCE_FILE: {payload.get("_source_file")}
SOURCE_FOLDER: {payload.get("_source_folder")}
SOURCE_TYPE: {payload.get("source_type")}
AUTHORITY: {payload.get("authority_level")}
CHUNK_TYPE: {payload.get("chunk_type")}
TEXT:
{payload.get("text")}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

### 15. Prueba final: contexto recuperado

Este es el contexto que después se podría pasar al LLM para generar la respuesta final del avatar.


In [ ]:
query = "¿Qué evidencia tiene SECURE?"
context = build_context(query, limit=3)

print(context[:3000])